# Rap-Snacks Boltz Validation v2

Folds four sequence buckets through Boltz-2 to validate structure vs scrambled control.

| Bucket | n | Label |
|--------|---|-------|
| Concordance originals | 12 | `concordance` |
| Native-alanine originals | 12 | `native_ala` |
| Free-design MPNN (top-5 diverse/bar) | 60 | `free_design` |
| Scrambled controls (5/bar) | 60 | `scrambled` |

**Expected result:** `scrambled` ≪ `concordance` ≈ `native_ala` < `free_design`

**Runtime:** ~10–15 min on A100 (144 sequences, 1 model each, `--no-kernels`)

---
## Cell 1 — Install

Run once per session. No restart needed.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', 'boltz'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'biopython'], check=True)

print('Boltz + BioPython installed.')

---
## Cell 2 — Mount Drive + Paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT    = Path('/content/drive/MyDrive/rap_snacks')
DRIVE_IN      = DRIVE_ROOT / 'inputs'
DRIVE_RES     = DRIVE_ROOT / 'results'
DRIVE_FIGS    = DRIVE_RES  / 'figures'
CONTENT       = Path('/content/scratch')

for p in [DRIVE_IN, DRIVE_RES, DRIVE_FIGS, CONTENT]:
    p.mkdir(parents=True, exist_ok=True)

DATA          = DRIVE_IN / 'data'
MPNN_FREE     = DRIVE_IN / 'outputs' / 'proteinmpnn_free' / 'filtered_results.csv'
SCRAMBLED     = DRIVE_IN / 'outputs' / 'scrambled'        / 'scrambled_results.csv'
BACKBONE_PDBS = DRIVE_IN / 'outputs' / 'proteinmpnn'      / 'pdbs'

for p in [DATA / 'aggregated_lines_v2_enriched.csv',
          DATA / 'phase2_candidates.csv',
          MPNN_FREE, SCRAMBLED, BACKBONE_PDBS]:
    status = '✅' if p.exists() else '❌ MISSING'
    print(f'{status}  {p.relative_to(DRIVE_ROOT)}')

---
## Cell 3 — Config

In [ ]:
import json, random, subprocess, sys, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_N       = 12
N_SCRAMBLES = 5     # scrambled seqs per bar
SEED        = 42
N_MODELS    = 5     # Boltz diffusion samples per sequence (averaged for confidence)

BOLTZ_IN    = CONTENT / 'boltz_inputs'
BOLTZ_OUT   = CONTENT / 'boltz_outputs'

VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

# 696 seqs x 5 models = 3480 predictions — ~2.5 hrs on A100
print('Config loaded. NOTE: 696 seqs x 5 models ~ 2.5 hrs on A100.')

---
## Cell 4 — Helpers

In [ ]:
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))


def greedy_diverse(seqs, n=5):
    """Greedy max-min Hamming: pick n sequences maximally spread in sequence space."""
    selected = [0]
    while len(selected) < min(n, len(seqs)):
        best, best_dist = -1, -1
        for i in range(len(seqs)):
            if i in selected: continue
            min_dist = min(hamming(seqs[i], seqs[j]) for j in selected)
            if min_dist > best_dist:
                best, best_dist = i, min_dist
        selected.append(best)
    return selected


def write_boltz_yaml(name, sequence, out_dir):
    """Write a single-chain Boltz input YAML."""
    path = out_dir / f'{name}.yaml'
    path.write_text(
        f'version: 1\n'
        f'sequences:\n'
        f'  - protein:\n'
        f'      id: A\n'
        f'      sequence: {sequence}\n'
    )
    return path


def parse_boltz_outputs(boltz_out_dir):
    """
    Parse Boltz output directory. Averages metrics across all diffusion models.
    Returns {name: {'plddt': float, 'ptm': float, 'confidence': float,
                    'plddt_std': float, 'n_models': int}}.
    pLDDT npz is per-residue 0-1 scale; confidence JSON has ptm + confidence_score.
    """
    results = {}
    preds_dir = Path(boltz_out_dir) / 'predictions'
    if not preds_dir.exists():
        print(f'[WARN] No predictions dir at {preds_dir}')
        return results

    for seq_dir in sorted(preds_dir.iterdir()):
        name = seq_dir.name
        conf_files  = sorted(seq_dir.glob('confidence_*.json'))
        plddt_files = sorted(seq_dir.glob('plddt_*.npz'))
        if not conf_files:
            continue
        plddt_means, ptms, confs = [], [], []
        for cf, pf in zip(conf_files, plddt_files):
            c = json.loads(cf.read_text())
            ptms.append(c.get('ptm', 0))
            confs.append(c.get('confidence_score', 0))
            arr = np.load(pf)
            key = list(arr.keys())[0]
            plddt_means.append(float(np.mean(arr[key])))
        results[name] = {
            'plddt':      float(np.mean(plddt_means)),
            'plddt_std':  float(np.std(plddt_means)),
            'ptm':        float(np.mean(ptms)),
            'confidence': float(np.mean(confs)),
            'n_models':   len(conf_files),
        }
    return results



def get_ca_atoms(pdb_path):
    from Bio.PDB import PDBParser
    parser = PDBParser(QUIET=True)
    st = parser.get_structure('s', str(pdb_path))
    return [r['CA'] for m in st for c in m for r in c if 'CA' in r]


def compute_rmsd(pdb_a, pdb_b):
    from Bio.PDB import Superimposer
    try:
        atoms_a = get_ca_atoms(pdb_a)
        atoms_b = get_ca_atoms(pdb_b)
        if len(atoms_a) != len(atoms_b) or not atoms_a:
            return None
        sup = Superimposer()
        sup.set_atoms(atoms_a, atoms_b)
        return round(float(sup.rms), 3)
    except Exception as e:
        return None


print('Helpers loaded.')

---
## Cell 5 — Build Boltz Inputs

Loads all four buckets and writes one YAML per sequence.

In [ ]:
BOLTZ_IN.mkdir(parents=True, exist_ok=True)

enriched   = pd.read_csv(DATA / 'aggregated_lines_v2_enriched.csv')
candidates = pd.read_csv(DATA / 'phase2_candidates.csv').head(TOP_N)
bar_ids    = list(candidates['bar_id'])
sub        = enriched[enriched['bar_id'].isin(bar_ids)].set_index('bar_id')

free_df    = pd.read_csv(MPNN_FREE)
sc_df      = pd.read_csv(SCRAMBLED)

meta_rows  = []

for bar_id in bar_ids:

    # concordance
    conc_seq = sub.loc[bar_id, 'fasta_seq_concordance']
    if pd.notna(conc_seq) and set(conc_seq.upper()) <= VALID_AA:
        name = f'{bar_id}_concordance'
        write_boltz_yaml(name, conc_seq, BOLTZ_IN)
        meta_rows.append((name, conc_seq, bar_id, 'concordance', ''))

    # native_ala
    na_seq = sub.loc[bar_id, 'fasta_seq_native_alanine']
    if pd.notna(na_seq) and set(na_seq.upper()) <= VALID_AA:
        name = f'{bar_id}_native_ala'
        write_boltz_yaml(name, na_seq, BOLTZ_IN)
        meta_rows.append((name, na_seq, bar_id, 'native_ala', ''))

    # free_design — ALL sequences
    grp = free_df[free_df['bar_id'] == bar_id].reset_index(drop=True)
    for i, row in grp.iterrows():
        if not set(row['sequence'].upper()) <= VALID_AA:
            continue
        name = f'{bar_id}_free_{i:03d}'
        write_boltz_yaml(name, row['sequence'], BOLTZ_IN)
        meta_rows.append((name, row['sequence'], bar_id, 'free_design',
                          f'esm_plddt={row["esm_plddt"]:.3f}'))

    # scrambled — 5 per bar
    sc_bar = sc_df[
        (sc_df['bar_id'] == bar_id) &
        (sc_df['source'] == 'scrambled_concordance') &
        (sc_df['seq'].notna())
    ].head(N_SCRAMBLES)
    for j, (_, sc_row) in enumerate(sc_bar.iterrows()):
        seq = sc_row['seq']
        if set(seq.upper()) <= VALID_AA:
            name = f'{bar_id}_scrambled_{j:02d}'
            write_boltz_yaml(name, seq, BOLTZ_IN)
            meta_rows.append((name, seq, bar_id, 'scrambled', ''))

meta = pd.DataFrame(meta_rows, columns=['name', 'sequence', 'bar_id', 'bucket', 'detail'])
print(f'Total sequences: {len(meta)}')
print(meta.groupby('bucket').size().rename('count').to_string())
print(f'YAML files written -> {BOLTZ_IN}')
print(f'File count: {len(list(BOLTZ_IN.glob("*.yaml")))}')

---
## Cell 6 — Run Boltz

`--no-kernels` is required on Colab (cuequivariance_ops_torch not available).

~10–15 min on A100.

In [ ]:
BOLTZ_OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    'boltz', 'predict', str(BOLTZ_IN),
    '--out_dir',         str(BOLTZ_OUT),
    '--diffusion_samples', str(N_MODELS),
    '--no-kernels',
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode != 0:
    print(f'Boltz exited with code {result.returncode}')
else:
    print('Boltz complete.')

---
## Cell 7 — Parse Results + Save to Drive

In [ ]:
scores = parse_boltz_outputs(BOLTZ_OUT)
print(f'Parsed {len(scores)} predictions')

meta['boltz_plddt']      = meta['name'].map(lambda n: scores.get(n, {}).get('plddt'))
meta['boltz_plddt_std']  = meta['name'].map(lambda n: scores.get(n, {}).get('plddt_std'))
meta['boltz_ptm']        = meta['name'].map(lambda n: scores.get(n, {}).get('ptm'))
meta['boltz_confidence'] = meta['name'].map(lambda n: scores.get(n, {}).get('confidence'))
meta['n_models']         = meta['name'].map(lambda n: scores.get(n, {}).get('n_models'))

results_path = CONTENT / 'boltz_validation_results.csv'
meta.to_csv(results_path, index=False)
shutil.copy2(results_path, DRIVE_RES / 'boltz_validation_results.csv')
print('Saved -> Drive: results/boltz_validation_results.csv')

print('--- Boltz pLDDT by bucket (mean across 5 models) ---')
for bucket, grp in meta.groupby('bucket'):
    vals = grp['boltz_plddt'].dropna()
    if vals.empty:
        print(f'  {bucket:15s}  no data')
        continue
    print(f'  {bucket:15s}  n={len(vals):3d}  mean={vals.mean():.3f}  '
          f'sd={vals.std():.3f}  min={vals.min():.3f}  max={vals.max():.3f}')

---
## Cell 9 — RMSD Analysis

- RMSD of each folded sequence vs concordance backbone
- Pairwise RMSD between the 5 free_design structures per bar

**Requires:** `inputs/outputs/proteinmpnn/pdbs/` on Drive (the 12 chain-fixed backbone PDBs)

In [ ]:
preds_dir = BOLTZ_OUT / 'predictions'
rmsd_rows = []

for bar_id in sorted(meta['bar_id'].unique()):
    backbone_pdb = BACKBONE_PDBS / f'{bar_id}.pdb'
    if not backbone_pdb.exists():
        print(f'  [SKIP] {bar_id} — no backbone PDB on Drive')
        continue

    bar_seqs = meta[meta['bar_id'] == bar_id]

    # RMSD vs backbone for every sequence
    free_pdbs = []
    for _, row in bar_seqs.iterrows():
        seq_dir  = preds_dir / row['name']
        pdb_list = sorted(seq_dir.glob('*.pdb')) if seq_dir.exists() else []
        if not pdb_list: continue
        seq_pdb = pdb_list[0]
        rmsd = compute_rmsd(backbone_pdb, seq_pdb)
        rmsd_rows.append({'name': row['name'], 'bar_id': bar_id,
                          'bucket': row['bucket'], 'rmsd_vs_backbone': rmsd})
        if row['bucket'] == 'free_design':
            free_pdbs.append((row['name'], seq_pdb))

    # Pairwise RMSD between free_design structures within the bar
    for i in range(len(free_pdbs)):
        for j in range(i+1, len(free_pdbs)):
            name_i, pdb_i = free_pdbs[i]
            name_j, pdb_j = free_pdbs[j]
            rmsd = compute_rmsd(pdb_i, pdb_j)
            rmsd_rows.append({'name': f'{name_i}_vs_{name_j}', 'bar_id': bar_id,
                              'bucket': 'free_pairwise', 'rmsd_vs_backbone': rmsd})

rmsd_df = pd.DataFrame(rmsd_rows)
rmsd_path = CONTENT / 'boltz_rmsd.csv'
rmsd_df.to_csv(rmsd_path, index=False)
shutil.copy2(rmsd_path, DRIVE_RES / 'boltz_rmsd.csv')
print('Saved -> Drive: results/boltz_rmsd.csv')

print('--- RMSD vs concordance backbone ---')
for bucket, grp in rmsd_df[rmsd_df['bucket'] != 'free_pairwise'].groupby('bucket'):
    vals = grp['rmsd_vs_backbone'].dropna()
    if vals.empty: continue
    print(f'  {bucket:15s}  n={len(vals):3d}  mean={vals.mean():.2f}A  '
          f'sd={vals.std():.2f}  min={vals.min():.2f}  max={vals.max():.2f}')

pw = rmsd_df[rmsd_df['bucket'] == 'free_pairwise']['rmsd_vs_backbone'].dropna()
if not pw.empty:
    print(f'  free_pairwise    n={len(pw):3d}  mean={pw.mean():.2f}A  '
          f'sd={pw.std():.2f}  min={pw.min():.2f}  max={pw.max():.2f}')

---
## Cell 10 — RMSD Figures

In [ ]:
DARK_BG = '#0e0e0e'
RMSD_COLORS = {'concordance':'#00d4ff','native_ala':'#ff9500',
               'free_design':'#a855f7','scrambled':'#444455'}
RMSD_ORDER  = ['concordance','native_ala','free_design','scrambled']

vs_bb = rmsd_df[rmsd_df['bucket'] != 'free_pairwise']

# Fig C: violin — RMSD vs backbone by bucket
fig3, ax3 = plt.subplots(figsize=(7, 5), facecolor=DARK_BG)
ax3.set_facecolor(DARK_BG)
valid = [(b, vs_bb[vs_bb['bucket']==b]['rmsd_vs_backbone'].dropna().values)
         for b in RMSD_ORDER if b in vs_bb['bucket'].values]
valid = [(b, d) for b, d in valid if len(d) > 1]
if valid:
    parts3 = ax3.violinplot([d for _, d in valid], positions=range(len(valid)),
                             showmedians=True, showextrema=False)
    for pc, (b, _) in zip(parts3['bodies'], valid):
        pc.set_facecolor(RMSD_COLORS.get(b, '#888')); pc.set_alpha(0.7)
    parts3['cmedians'].set_color('white')
ax3.set_xticks(range(len(valid)))
ax3.set_xticklabels([b for b, _ in valid], color='white', fontsize=10)
ax3.set_ylabel('RMSD vs concordance backbone (Å)', color='white')
ax3.set_title('Backbone deviation by bucket', color='white', fontsize=12)
ax3.tick_params(colors='white')
for s in ax3.spines.values(): s.set_edgecolor('#333')
plt.tight_layout()
fig_c = CONTENT / 'fig_rmsd_violin.png'
plt.savefig(fig_c, dpi=150, facecolor=DARK_BG)
shutil.copy2(fig_c, DRIVE_FIGS / 'fig_rmsd_violin.png')
plt.show(); print('Fig C saved.')

# Fig D: per-bar free_design RMSD vs backbone
free_rmsd = vs_bb[vs_bb['bucket'] == 'free_design']
bar_ids_s = sorted(free_rmsd['bar_id'].unique())
fig4, ax4 = plt.subplots(figsize=(12, 4), facecolor=DARK_BG)
ax4.set_facecolor(DARK_BG)
for bi, bar_id in enumerate(bar_ids_s):
    vals = free_rmsd[free_rmsd['bar_id']==bar_id]['rmsd_vs_backbone'].dropna()
    ax4.scatter([bi]*len(vals), vals, color='#a855f7', s=45, alpha=0.85, zorder=3)
    if not vals.empty:
        ax4.plot([bi-0.2,bi+0.2],[vals.mean(),vals.mean()],
                 color='white', linewidth=2.5, zorder=4)
ax4.set_xticks(range(len(bar_ids_s)))
ax4.set_xticklabels(bar_ids_s, rotation=45, ha='right', fontsize=8, color='white')
ax4.set_ylabel('RMSD vs concordance backbone (Å)', color='white')
ax4.set_title('free_design RMSD vs backbone — per bar', color='white', fontsize=12)
ax4.tick_params(colors='white')
for s in ax4.spines.values(): s.set_edgecolor('#333')
plt.tight_layout()
fig_d = CONTENT / 'fig_rmsd_per_bar.png'
plt.savefig(fig_d, dpi=150, facecolor=DARK_BG)
shutil.copy2(fig_d, DRIVE_FIGS / 'fig_rmsd_per_bar.png')
plt.show(); print('Fig D saved.')

# Fig E: pairwise RMSD distribution (free_design vs free_design within bar)
pw_vals = rmsd_df[rmsd_df['bucket']=='free_pairwise']['rmsd_vs_backbone'].dropna()
if not pw_vals.empty:
    fig5, ax5 = plt.subplots(figsize=(6, 4), facecolor=DARK_BG)
    ax5.set_facecolor(DARK_BG)
    ax5.hist(pw_vals, bins=20, color='#a855f7', alpha=0.8, edgecolor='#333')
    ax5.axvline(pw_vals.mean(), color='white', linestyle='--', linewidth=1.5,
                label=f'mean={pw_vals.mean():.2f}Å')
    ax5.set_xlabel('Pairwise RMSD (Å)', color='white')
    ax5.set_ylabel('Count', color='white')
    ax5.set_title('free_design structural diversity (pairwise RMSD within bar)',
                  color='white', fontsize=11)
    ax5.tick_params(colors='white')
    for s in ax5.spines.values(): s.set_edgecolor('#333')
    ax5.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=9)
    plt.tight_layout()
    fig_e = CONTENT / 'fig_rmsd_pairwise.png'
    plt.savefig(fig_e, dpi=150, facecolor=DARK_BG)
    shutil.copy2(fig_e, DRIVE_FIGS / 'fig_rmsd_pairwise.png')
    plt.show(); print('Fig E saved.')

---
## Cell 8 — Figures

In [ ]:
DARK_BG = '#0e0e0e'
COLORS  = {
    'concordance': '#00d4ff',
    'native_ala':  '#ff9500',
    'free_design': '#a855f7',
    'scrambled':   '#444455',
}
ORDER = ['scrambled', 'concordance', 'native_ala', 'free_design']

# ── Fig A: strip plot per bar ─────────────────────────────────────────────
bar_ids_sorted = sorted(meta['bar_id'].unique())
n_bars  = len(bar_ids_sorted)
n_buck  = len(ORDER)
width   = 0.18
offsets = np.linspace(-(n_buck-1)*width/2, (n_buck-1)*width/2, n_buck)

fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)
plotted = set()

for bi, bar_id in enumerate(bar_ids_sorted):
    for oi, bucket in enumerate(ORDER):
        vals = meta[(meta['bar_id']==bar_id) & (meta['bucket']==bucket)]['boltz_plddt'].dropna()
        if vals.empty: continue
        x = bi + offsets[oi]
        label = bucket if bucket not in plotted else ''
        ax.scatter([x]*len(vals), vals, color=COLORS[bucket], s=30, alpha=0.8,
                   label=label, zorder=3)
        ax.plot([x-0.05, x+0.05], [vals.mean(), vals.mean()],
                color=COLORS[bucket], linewidth=2.5, zorder=4)
        plotted.add(bucket)

ax.axhline(0.5, color='white', linestyle='--', linewidth=0.8, alpha=0.4)
ax.set_xticks(range(n_bars))
ax.set_xticklabels(bar_ids_sorted, rotation=45, ha='right', fontsize=8, color='white')
ax.set_ylabel('Boltz pLDDT (0–1)', color='white')
ax.set_ylim(0, 1)
ax.set_title('Boltz-2 pLDDT: free_design vs concordance vs native_ala vs scrambled',
             color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#333333')
handles, labels = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels):
    if l and l not in seen: seen[l] = h
ax.legend(seen.values(), seen.keys(), facecolor='#1a1a1a',
          labelcolor='white', framealpha=0.8, fontsize=9)
plt.tight_layout()
fig_a = CONTENT / 'fig_boltz_per_bar.png'
plt.savefig(fig_a, dpi=150, facecolor=DARK_BG)
shutil.copy2(fig_a, DRIVE_FIGS / 'fig_boltz_per_bar.png')
plt.show()
print('Fig A saved.')

# ── Fig B: violin per bucket ──────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 5), facecolor=DARK_BG)
ax2.set_facecolor(DARK_BG)
data_by_bucket = [meta[meta['bucket']==b]['boltz_plddt'].dropna().values for b in ORDER]
parts = ax2.violinplot(data_by_bucket, positions=range(len(ORDER)),
                       showmedians=True, showextrema=False)
for pc, bucket in zip(parts['bodies'], ORDER):
    pc.set_facecolor(COLORS[bucket])
    pc.set_alpha(0.7)
parts['cmedians'].set_color('white')
ax2.set_xticks(range(len(ORDER)))
ax2.set_xticklabels(ORDER, color='white', fontsize=10)
ax2.set_ylabel('Boltz pLDDT (0–1)', color='white')
ax2.set_ylim(0, 1)
ax2.set_title('Boltz-2 pLDDT distribution by bucket', color='white', fontsize=12)
ax2.tick_params(colors='white')
for spine in ax2.spines.values(): spine.set_edgecolor('#333333')
ax2.axhline(0.5, color='white', linestyle='--', linewidth=0.8, alpha=0.4)
plt.tight_layout()
fig_b = CONTENT / 'fig_boltz_violin.png'
plt.savefig(fig_b, dpi=150, facecolor=DARK_BG)
shutil.copy2(fig_b, DRIVE_FIGS / 'fig_boltz_violin.png')
plt.show()
print('Fig B saved.')